# Executable engineering production-qualification workflow

This notebook demonstrates the benchmark, named-tool DEXPI, pilot, release and governed relief-sizing APIs added around the process-to-engineering simulator. All references and values below are **synthetic API examples**. A passing cell is not project evidence and does not make a package fit for construction.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=not (ROOT / 'target/classes').exists(), verbose=False))
import jpype
JClass = ns.JClass
HashMap = JClass('java.util.LinkedHashMap')
ArrayList = JClass('java.util.ArrayList')
print('Loaded qualification API from', ROOT)


## 1. Run exact-version benchmark data

Expected values, actual simulator outputs, tolerances, source class and independent-review record stay distinct. Missing actual outputs fail the run instead of being replaced by expected values.


In [ ]:
Dataset = JClass('neqsim.process.engineering.production.EngineeringBenchmarkDataset')
Case = JClass('neqsim.process.engineering.production.EngineeringBenchmarkDataset$Case')
SourceClass = JClass('neqsim.process.engineering.production.EngineeringValidationBenchmark$SourceClass')
dataset = Dataset('synthetic-separator-reference', 'A')
case = Case('SEP-CASE-1', 'separator-scrubber-preliminary-design', '2.0',
            SourceClass.INDEPENDENT_CALCULATION, 'SYNTHETIC-CALC-001', 'A',
            'SYNTHETIC-INDEPENDENT-REVIEW')
case.expect('insideDiameterM', 2.0, 'm', 0.02, 0.01)
dataset.add(case)
case_outputs = HashMap(); case_outputs.put('insideDiameterM', 2.01)
actual_outputs = HashMap(); actual_outputs.put('SEP-CASE-1', case_outputs)
benchmark_run = dataset.run(actual_outputs)
print('complete=', benchmark_run.isComplete(), 'passed=', benchmark_run.getReport().isPassed())


## 2. Compare named-tool DEXPI semantics

In a real qualification, importer adapters create the two post-tool snapshots from a named CAE product. Qualification requires zero object, property and relationship differences after import and export/reimport.


In [ ]:
Dexpi = JClass('neqsim.process.engineering.production.DexpiToolQualificationRunner')
Snapshot = JClass('neqsim.process.engineering.production.DexpiToolQualificationRunner$Snapshot')
properties = HashMap(); properties.put('tag', '20-VA-001'); properties.put('class', 'Vessel')
reference = Snapshot('DEXPI 2.0').addObject('urn:equipment:1', properties)
after_import = Snapshot('DEXPI 2.0').addObject('urn:equipment:1', properties)
after_roundtrip = Snapshot('DEXPI 2.0').addObject('urn:equipment:1', properties)
dexpi_result = Dexpi.compare('SYNTHETIC-CAE', '2026.1', reference, after_import, after_roundtrip,
                             'SYNTHETIC-DEXPI-EVIDENCE', 'SYNTHETIC-CHECKER')
print('qualified=', dexpi_result.isQualified())


## 3. Derive pilot and release gates from measurements

Material pilot discrepancies block acceptance. Release booleans are derived from CI, Java, repeatability, performance, API, migration and security measurements and require an accountable reviewer.


In [ ]:
Pilot = JClass('neqsim.process.engineering.production.EngineeringPilotQualificationRunner')
Comparison = JClass('neqsim.process.engineering.production.EngineeringPilotQualificationRunner$Comparison')
Scope = JClass('neqsim.process.engineering.production.EngineeringPilotProjectEvidence$Scope')
comparisons = ArrayList(); comparisons.add(Comparison('separator diameter', 2.0, 2.01, 'm', 0.02, 0.01, True))
pilot = Pilot.run('SYNTHETIC-PILOT', Scope.SEPARATION_AND_COMPRESSION, 'SYNTHETIC-REFERENCE',
                  comparisons, 'SYNTHETIC-CHECKER', 'SYNTHETIC-ACCEPTANCE')
ReleaseInput = JClass('neqsim.process.engineering.production.EngineeringReleaseQualificationRunner$Input')
ReleaseRunner = JClass('neqsim.process.engineering.production.EngineeringReleaseQualificationRunner')
release_input = (ReleaseInput('2.0-rc1', 'SYNTHETIC-CI-RUN', 'SYNTHETIC-RELEASE-AUTHORITY')
                 .fullCiPassed(True).requireJavaVersion('8').passedJavaVersion('8')
                 .deterministicFingerprint('sha256-a').deterministicFingerprint('sha256-a')
                 .performanceSample(2.0).performanceSample(2.2).maximumAcceptedSeconds(5.0)
                 .apiFingerprints('api-a', 'api-a').serializationMigrationPassed(True)
                 .openHighSeveritySecurityFindings(0))
release = ReleaseRunner.run(release_input)
print('pilot accepted=', pilot.getEvidence().isAccepted(), 'release passed=', release.getEvidence().isPassed())


## 4. Use fail-closed relief sizing

The gas example uses explicit absolute pressures, thermodynamic properties, correction factors and a controlled orifice table. Two-phase flow is blocked unless a specialist mass-flux value and controlled method reference are supplied.


In [ ]:
Context = JClass('neqsim.process.engineering.calculation.EngineeringCalculationContext')
Relief = JClass('neqsim.process.engineering.safety.ReliefSizingCalculation')
ReliefInput = JClass('neqsim.process.engineering.safety.ReliefSizingCalculation$Input')
Phase = JClass('neqsim.process.engineering.safety.ReliefSizingCalculation$Phase')
context = (Context.builder().designCaseId('blocked-outlet')
           .addStandardReference('SYNTHETIC-PROJECT-RELIEF-BASIS')
           .addEvidenceReference('SYNTHETIC-RELIEF-CHECK')
           .attribute('productionQualification', 'true').build())
orifices = jpype.JArray(jpype.JDouble)([0.11, 0.196, 0.307, 0.503, 0.785, 1.287])
gas = (ReliefInput.builder('blocked-outlet', Phase.GAS).flowAndPressure(2.0, 60.0, 2.0)
       .gasProperties(320.0, 1.28, 0.95, 20.0).correctionFactors(0.975, 1.0, 1.0)
       .orificeCandidatesIn2(orifices).build())
relief_result = Relief().calculate(gas, context)
print(relief_result.getStatus(), relief_result.getValue().toMap())


## Package handoff

`EngineeringDeliverableCompiler` now writes `engineering-qualification-plan.json` beside `engineering-production-readiness.json`. Work the open plan actions using controlled sources, rerun the assessment, and retain both artifacts in the revisioned package. Even `QUALIFIED_FEED_SUPPORT` remains preliminary, review-required and explicitly not fit for construction.
